# Objects365 Duck Dataset Extractor for Duck Counting

This notebook extracts **only Duck instances** from the [Objects365 dataset](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/datasets/Objects365.yaml) for a specialized computer vision task: **Duck Counting**.

### Why Selective Extraction?
The full Objects365 v2 dataset is **over 700 GB** (1.7M images across 51 train patches and 44 validation patches).
Downloading the full dataset just to get duck images would waste hundreds of gigabytes of disk and bandwidth.

**Our Selective Pipeline:**
1. **Download the annotation JSON** (~269 MB for validation).
2. **Filter Duck annotations** (Objects365 v2 Category ID: `137`).
3. **Map Duck images to specific patch archives** (`patchX.tar.gz`).
4. **Selective Stream & Extract**: Download only the patches containing duck images, extract *strictly the duck files*, and discard the rest of the archive immediately.
5. **Generate YOLO annotations**: Convert COCO bounding boxes `[x, y, w, h]` into normalized YOLO format `[0, x_center, y_center, w, h]`.
6. **Analyze Duck Count Distribution**: Generate statistics and histograms for duck counting.
7. **Configure & Train YOLO for Duck Counting**: Generate `duck.yaml` and provide model training code with Ultralytics YOLOv8 / YOLO11.

In [7]:
import os
import sys
import json
import shutil
import tarfile
from pathlib import Path
from collections import defaultdict
from urllib.request import Request, urlopen

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Verify working directory
WORK_DIR = Path(".").resolve()
print(f"Working Directory: {WORK_DIR}")

Working Directory: D:\DataNLP\duck


## 1. Dataset Configuration

- `BASE_URL`: Megvii / KSYun repository for Objects365 v2.
- `SPLIT`: We begin with `'val'` (80,000 images, ~1,547 duck images with 12,221 duck instances). You can also set this to `'train'` for the full set.
- `DUCK_CAT_ID`: `137` is the official Objects365 v2 category ID for `Duck`.

In [8]:
BASE_URL = "https://dorc.ks3-cn-beijing.ksyun.com/data-set/2020Objects365%E6%95%B0%E6%8D%AE%E9%9B%86"

# Choose split: 'val' (recommended to start, fast) or 'train'
SPLIT = 'val'
DUCK_CATEGORY_ID = 137  # 'Duck' in Objects365 v2

# Directory paths
IMAGES_DIR = WORK_DIR / "images" / SPLIT
LABELS_DIR = WORK_DIR / "labels" / SPLIT
TEMP_DIR = WORK_DIR / "temp_downloads"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Images dir: {IMAGES_DIR}")
print(f"Labels dir: {LABELS_DIR}")

Images dir: D:\DataNLP\duck\images\val
Labels dir: D:\DataNLP\duck\labels\val


## 2. Download and Parse Objects365 Annotations

The validation annotations file `zhiyuan_objv2_val.json` is ~269 MB and contains metadata for 80,000 images and 1.2M annotations.
We download it with chunked progress reporting.

In [9]:
def download_with_progress(url: str, dest_path: Path, chunk_size: int = 1024 * 1024):
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.exists():
        print(f"File already exists: {dest_path.name} ({dest_path.stat().st_size / 1e6:.1f} MB)")
        return

    print(f"Downloading {url} -> {dest_path.name}...")
    req = Request(url, headers={"User-Agent": "Objects365-DuckExtractor/1.0"})
    with urlopen(req) as resp, open(dest_path, "wb") as f:
        total = int(resp.headers.get("content-length", 0))
        downloaded = 0
        while True:
            chunk = resp.read(chunk_size)
            if not chunk:
                break
            f.write(chunk)
            downloaded += len(chunk)
            if downloaded % (15 * 1024 * 1024) < chunk_size:
                pct = (downloaded / total * 100) if total else 0
                print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB ({pct:.1f}%)")
    print(f"Done downloading {dest_path.name}!")

# Annotation file download
if SPLIT == 'val':
    ann_url = f"{BASE_URL}/val/zhiyuan_objv2_val.json"
    ann_file = WORK_DIR / "zhiyuan_objv2_val.json"
    download_with_progress(ann_url, ann_file)
else:
    ann_url = f"{BASE_URL}/train/zhiyuan_objv2_train.tar.gz"
    ann_tar = WORK_DIR / "zhiyuan_objv2_train.tar.gz"
    ann_file = WORK_DIR / "zhiyuan_objv2_train.json"
    if not ann_file.exists():
        download_with_progress(ann_url, ann_tar)
        print("Unpacking train annotations tar.gz...")
        with tarfile.open(ann_tar, "r:gz") as tar:
            tar.extractall(path=WORK_DIR)

print(f"Annotation file ready: {ann_file}")

File already exists: zhiyuan_objv2_val.json (269.4 MB)
Annotation file ready: D:\DataNLP\duck\zhiyuan_objv2_val.json


## 3. Filter Duck Annotations

We parse the JSON file, locate the `Duck` category (Category ID 137), and filter every image that contains one or more duck bounding boxes.

In [10]:
print("Loading JSON annotations (this takes ~10-15 seconds)...")
with open(ann_file, "r", encoding="utf-8") as f:
    raw_coco = json.load(f)

categories = {cat["id"]: cat["name"] for cat in raw_coco.get("categories", [])}
print(f"Target Category: ID {DUCK_CATEGORY_ID} -> {categories.get(DUCK_CATEGORY_ID)}")

# Index all images by id
images_by_id = {img["id"]: img for img in raw_coco["images"]}

# Filter annotations belonging to Duck
duck_annotations_by_img = defaultdict(list)
for ann in raw_coco["annotations"]:
    if ann.get("category_id") == DUCK_CATEGORY_ID:
        duck_annotations_by_img[ann["image_id"]].append(ann)

# Filter image objects that contain ducks
duck_images = {img_id: images_by_id[img_id] for img_id in duck_annotations_by_img if img_id in images_by_id}
total_ducks = sum(len(anns) for anns in duck_annotations_by_img.values())

print(f"Total Images with Ducks: {len(duck_images):,}")
print(f"Total Duck Bounding Boxes: {total_ducks:,}")
print(f"Average Ducks per Image: {total_ducks / len(duck_images):.2f}")

Loading JSON annotations (this takes ~10-15 seconds)...
Target Category: ID 137 -> Duck
Total Images with Ducks: 1,547
Total Duck Bounding Boxes: 12,221
Average Ducks per Image: 7.90


## 4. Map Images to Objects365 Archive Patches

Objects365 images are stored in tar.gz patches:
- `val/images/v1/patch0.tar.gz` ... `patch15.tar.gz` (for v1 patches)
- `val/images/v2/patch16.tar.gz` ... `patch43.tar.gz` (for v2 patches)

By mapping each duck image to its patch, we only download patches that contain duck images.

In [11]:
def resolve_patch(file_name: str, split: str = 'val'):
    fn = file_name.replace("\\", "/")
    if split == 'val':
        if "v1/patch" in fn:
            p_num = fn.split("v1/patch")[1].split("/")[0]
            return f"v1_patch{p_num}", f"{BASE_URL}/val/images/v1/patch{p_num}.tar.gz"
        elif "v2/patch" in fn:
            p_num = fn.split("v2/patch")[1].split("/")[0]
            return f"v2_patch{p_num}", f"{BASE_URL}/val/images/v2/patch{p_num}.tar.gz"
    else:
        p_num = fn.split("patch")[1].split("/")[0]
        return f"patch{p_num}", f"{BASE_URL}/train/patch{p_num}.tar.gz"
    raise ValueError(f"Unknown patch format: {file_name}")

# Group images by patch
patch_map = defaultdict(lambda: {"url": "", "files": set()})
for img_id, img in duck_images.items():
    base_name = Path(img["file_name"]).name
    patch_key, patch_url = resolve_patch(img["file_name"], split=SPLIT)
    patch_map[patch_key]["url"] = patch_url
    patch_map[patch_key]["files"].add(base_name)

print(f"Duck images are distributed across {len(patch_map)} patches:")
for p_key in sorted(patch_map.keys())[:10]:
    print(f"  {p_key}: {len(patch_map[p_key]['files'])} duck images")
print(f"  ... ({len(patch_map) - 10} more patches)")

Duck images are distributed across 44 patches:
  v1_patch0: 27 duck images
  v1_patch1: 39 duck images
  v1_patch10: 35 duck images
  v1_patch11: 28 duck images
  v1_patch12: 20 duck images
  v1_patch13: 23 duck images
  v1_patch14: 22 duck images
  v1_patch15: 22 duck images
  v1_patch2: 25 duck images
  v1_patch3: 45 duck images
  ... (34 more patches)


## 5. Selective Extraction (Saves >95% Disk Space!)

We iterate through the required patches:
1. Download the patch `.tar.gz` into `temp_downloads/`.
2. Open the archive and extract **ONLY** the duck image files to `images/val/`.
3. Immediately delete the temporary `.tar.gz` to free up space.

> **Tip**: You can set `MAX_PATCHES_TO_PROCESS = None` to download all duck images, or set a number (e.g. `3`) to quickly test.

In [6]:
# Control how many patches to extract (None = extract all duck images)
MAX_PATCHES_TO_PROCESS = None  # Change to e.g. 5 for a quick test

processed_patches = 0
extracted_images = 0

for patch_key, p_info in sorted(patch_map.items()):
    if MAX_PATCHES_TO_PROCESS and processed_patches >= MAX_PATCHES_TO_PROCESS:
        print(f"Reached limit of {MAX_PATCHES_TO_PROCESS} patches.")
        break

    needed_files = p_info["files"]
    # Skip if all files for this patch already exist locally
    already_have = [f for f in needed_files if (IMAGES_DIR / f).exists()]
    if len(already_have) == len(needed_files):
        print(f"[{patch_key}] All {len(needed_files)} duck images already exist. Skipping.")
        continue

    tar_path = TEMP_DIR / f"{patch_key}.tar.gz"
    print(f"\nProcessing {patch_key} (needs {len(needed_files)} duck images)...")
    download_with_progress(p_info["url"], tar_path)

    # Extract only duck images
    count = 0
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar.getmembers():
            base_name = Path(member.name).name
            if base_name in needed_files:
                f_obj = tar.extractfile(member)
                if f_obj:
                    with open(IMAGES_DIR / base_name, "wb") as out_f:
                        shutil.copyfileobj(f_obj, out_f)
                    count += 1

    print(f"[{patch_key}] Extracted {count}/{len(needed_files)} duck images.")
    extracted_images += count
    processed_patches += 1

    # Clean up archive to keep disk usage minimal
    if tar_path.exists():
        tar_path.unlink()
        print(f"[{patch_key}] Deleted temporary archive {tar_path.name}")

print(f"\nTotal Duck images currently in {IMAGES_DIR}: {len(list(IMAGES_DIR.glob('*.jpg')))}")

[v1_patch0] All 27 duck images already exist. Skipping.

Processing v1_patch1 (needs 39 duck images)...
  15.7 / 130.2 MB (12.1%)
  31.5 / 130.2 MB (24.2%)
  47.2 / 130.2 MB (36.2%)
  62.9 / 130.2 MB (48.3%)
  78.6 / 130.2 MB (60.4%)
  94.4 / 130.2 MB (72.5%)
  110.1 / 130.2 MB (84.6%)
  125.8 / 130.2 MB (96.6%)
Done downloading v1_patch1.tar.gz!
[v1_patch1] Extracted 39/39 duck images.
[v1_patch1] Deleted temporary archive v1_patch1.tar.gz

Processing v1_patch10 (needs 35 duck images)...
  15.7 / 155.9 MB (10.1%)
  31.5 / 155.9 MB (20.2%)
  47.2 / 155.9 MB (30.3%)
  62.9 / 155.9 MB (40.4%)
  78.6 / 155.9 MB (50.5%)
  94.4 / 155.9 MB (60.5%)
  110.1 / 155.9 MB (70.6%)
  125.8 / 155.9 MB (80.7%)
  141.6 / 155.9 MB (90.8%)
Done downloading v1_patch10.tar.gz!
[v1_patch10] Extracted 35/35 duck images.
[v1_patch10] Deleted temporary archive v1_patch10.tar.gz

Processing v1_patch11 (needs 28 duck images)...
  15.7 / 243.7 MB (6.5%)
  31.5 / 243.7 MB (12.9%)
  47.2 / 243.7 MB (19.4%)
  62.9 /

EOFError: Compressed file ended before the end-of-stream marker was reached

## 6. Generate YOLO Format Annotations for Duck Counting

For each duck image, we convert COCO bounding boxes `[x, y, w, h]` into normalized YOLO coordinates:
$$\text{x\_center} = \frac{x + w / 2}{\text{width}}, \quad \text{y\_center} = \frac{y + h / 2}{\text{height}}, \quad \text{w\_norm} = \frac{w}{\text{width}}, \quad \text{h\_norm} = \frac{h}{\text{height}}$$

All bounding boxes are assigned **Class 0 (`duck`)**.

In [ ]:
counts_per_image = {}

for img_id, img_info in duck_images.items():
    base_name = Path(img_info["file_name"]).name
    txt_name = Path(base_name).with_suffix(".txt").name
    label_file = LABELS_DIR / txt_name

    img_w = img_info["width"]
    img_h = img_info["height"]
    anns = duck_annotations_by_img.get(img_id, [])
    counts_per_image[base_name] = len(anns)

    lines = []
    for ann in anns:
        x, y, w, h = ann["bbox"]
        xc = max(0.0, min(1.0, (x + w / 2.0) / img_w))
        yc = max(0.0, min(1.0, (y + h / 2.0) / img_h))
        wn = max(0.0, min(1.0, w / img_w))
        hn = max(0.0, min(1.0, h / img_h))
        lines.append(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

    with open(label_file, "w", encoding="utf-8") as f:
        f.writelines(lines)

print(f"Generated {len(counts_per_image)} YOLO label files in: {LABELS_DIR}")

## 7. Duck Counting Data Analysis & Distribution

In duck counting tasks, it's vital to know the density distribution:
- Single duck images
- Small groups (2-5 ducks)
- Dense flocks (10+ ducks)

In [ ]:
counts = list(counts_per_image.values())

plt.figure(figsize=(10, 4))
plt.hist(counts, bins=range(1, min(50, max(counts) + 2)), color='#2563eb', edgecolor='black', alpha=0.8)
plt.title(f"Duck Count Distribution per Image (Max: {max(counts)}, Mean: {np.mean(counts):.2f})", fontsize=13)
plt.xlabel("Number of Ducks in Image", fontsize=11)
plt.ylabel("Image Count", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("=== Duck Counting Breakdown ===")
print(f"1 duck:     {sum(1 for c in counts if c == 1):4d} images ({sum(1 for c in counts if c == 1)/len(counts)*100:.1f}%)")
print(f"2 ducks:    {sum(1 for c in counts if c == 2):4d} images ({sum(1 for c in counts if c == 2)/len(counts)*100:.1f}%)")
print(f"3-5 ducks:  {sum(1 for c in counts if 3 <= c <= 5):4d} images ({sum(1 for c in counts if 3 <= c <= 5)/len(counts)*100:.1f}%)")
print(f"6-10 ducks: {sum(1 for c in counts if 6 <= c <= 10):4d} images ({sum(1 for c in counts if 6 <= c <= 10)/len(counts)*100:.1f}%)")
print(f"10+ ducks:  {sum(1 for c in counts if c > 10):4d} images ({sum(1 for c in counts if c > 10)/len(counts)*100:.1f}%)")

## 8. Visualize Duck Ground Truth & Bounding Boxes

Let's visualize downloaded duck images with their ground truth bounding boxes and duck count banner.

In [ ]:
downloaded_imgs = list(IMAGES_DIR.glob("*.jpg"))
print(f"Locally available duck images: {len(downloaded_imgs)}")

if downloaded_imgs:
    sample_imgs = downloaded_imgs[:min(3, len(downloaded_imgs))]
    fig, axes = plt.subplots(1, len(sample_imgs), figsize=(16, 6))
    if len(sample_imgs) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, sample_imgs):
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        # Read YOLO label
        label_path = LABELS_DIR / img_path.with_suffix(".txt").name
        duck_count = 0
        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        duck_count += 1
                        xc, yc, bw, bh = map(float, parts[1:])
                        x1 = int((xc - bw / 2) * w)
                        y1 = int((yc - bh / 2) * h)
                        x2 = int((xc + bw / 2) * w)
                        y2 = int((yc + bh / 2) * h)
                        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)

        ax.imshow(img_rgb)
        ax.set_title(f"{img_path.name}\nGround Truth: {duck_count} Ducks", fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 9. Create `duck.yaml` Dataset Configuration

We generate `duck.yaml` for Ultralytics YOLO training.

In [ ]:
yaml_text = f"""# Ultralytics YOLO Duck Counting Dataset
path: {WORK_DIR.as_posix()}
train: images/val  # Set to images/val or split into train/val
val: images/val

names:
  0: duck
"""

yaml_file = WORK_DIR / "duck.yaml"
with open(yaml_file, "w", encoding="utf-8") as f:
    f.write(yaml_text)

print(f"Generated dataset config at: {yaml_file}")
print(yaml_text)

## 10. Train YOLO Model for Duck Counting

Now you can train a model (e.g. YOLOv8 or YOLO11) using the Ultralytics framework:
```bash
pip install ultralytics
yolo train data=duck.yaml model=yolo11n.pt epochs=50 imgsz=640
```

Below is the Python training and inference/counting snippet.

In [ ]:
# Training snippet (requires ultralytics installed: pip install ultralytics)
code_example = '''
from ultralytics import YOLO

# 1. Initialize pretrained lightweight model
model = YOLO('yolo11n.pt')

# 2. Train on our duck counting dataset
results = model.train(
    data='duck.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='duck_counter'
)

# 3. Predict & Count Ducks in any image
def count_ducks(image_path, model, conf=0.25):
    results = model(image_path, conf=conf)[0]
    duck_count = len(results.boxes)
    print(f"{image_path}: Found {duck_count} ducks!")
    return duck_count, results.plot()
'''
print(code_example)